<a href="https://colab.research.google.com/github/vinayagarwal5/fullStackDatascience/blob/main/SwingTrade.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
!pip uninstall -y textblob pandas
!pip install pandas numpy yfinance ta gradio vaderSentiment finvader

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 106.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.5 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.5 which is incompatible.
dask-cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.5 which is incompatible.


In [ ]:
import gradio as gr
import pandas as pd
import numpy as np
import yfinance as yf
import ta
from datetime import datetime, timedelta
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # Switched to vaderSentiment due to finvader import issues

# 1. Pipeline Engine: Analyze technical data and process mock news stream via FinVader
def scan_stocks_for_swing():
    # Target Indian Watchlist (Feel free to modify tickers; add '.NS' for NSE)
    tickers = ["RELIANCE.NS", "TCS.NS", "INFY.NS", "HDFCBANK.NS", "ICICIBANK.NS", "BHARTIARTL.NS", "SBI.NS"]

    end_date = datetime.now()
    start_date = end_date - timedelta(days=60) # Fetch 60 days of history for indicators

    results = []

    # Mock news generation (Simulating live scraped text per ticker)
    # In a real environment, you would scrape sites like Moneycontrol or Economic Times
    mock_news_db = {
        "RELIANCE.NS": ["Reliance Q3 earnings beat estimates, expansion accelerating.", "Retail business marks stellar growth footprint."],
        "TCS.NS": ["IT sector faces structural slowdown due to global IT spending cuts.", "TCS margin under pressure amid high attrition."],
        "INFY.NS": ["Infosys signs mega $2B digital transformation deal.", "Bullish guidance issued for coming quarters."],
        "HDFCBANK.NS": ["Net interest income grows steadily, asset quality remains strong.", "Analyst upgrade follows solid loan book metrics."],
        "ICICIBANK.NS": ["Credit growth accelerates, stock eyes key resistance levels.", "Bullish momentum builds post board meeting updates."],
        "BHARTIARTL.NS": ["Tariff hikes likely to bolster average revenue per user.", "5G infrastructure spending hits cash flow expectations."],
        "SBI.NS": ["Bad loan provisions fall significantly, treasury gains boost bottomline.", "Public sector bank rally slows down amid retail profit booking."]
    }

    # Initialize FinVader analyzer once per scan
    analyzer = SentimentIntensityAnalyzer() # Switched to SentimentIntensityAnalyzer

    for ticker in tickers:
        try:
            # --- A. Technical Scanning ---
            stock = yf.Ticker(ticker)
            df = stock.history(start=start_date, end=end_date)

            if len(df) < 20:
                continue

            # Calculate standard swing trading indicators using the 'ta' library
            df['RSI'] = ta.momentum.rsi(df['Close'], window=14)
            df['SMA_20'] = ta.trend.sma_indicator(df['Close'], window=20)
            df['SMA_50'] = ta.trend.sma_indicator(df['Close'], window=50)

            latest_close = float(df['Close'].iloc[-1])
            latest_rsi = float(df['RSI'].iloc[-1])
            prev_rsi = float(df['RSI'].iloc[-2])
            sma20 = float(df['SMA_20'].iloc[-1])
            sma50 = float(df['SMA_50'].iloc[-1])

            # --- B. Financial Sentiment Processing using VaderSentiment ---
            news_items = mock_news_db.get(ticker, ["Market trading continues standard volatile patterns."])

            all_sentiment_scores = []
            for news_text in news_items:
                sentiment_result = analyzer.polarity_scores(news_text) # Changed method call for vaderSentiment
                all_sentiment_scores.append(sentiment_result['compound'])

            sentiment_score = 0.0
            if all_sentiment_scores:
                sentiment_score = float(np.mean(all_sentiment_scores))

            # --- C. Swing Trading Logic Rules ---
            # Rule 1: RSI is oversold (< 40) but hooking up, OR stock is cross-confirming upward over 50
            # Rule 2: Sentiment must be positive (> 0.05)
            # Rule 3: Price is floating sustainably close to or above the 20 SMA

            status = "🔍 Neutral"
            score = 50

            if latest_rsi > prev_rsi and latest_rsi < 45 and sentiment_score > 0.1:
                status = "🚀 BUY WATCH (RSI Reversal + Bullish News)"
                score = 85
            elif latest_close > sma20 and sma20 > sma50 and sentiment_score > 0.2:
                status = "📈 BUY WATCH (Trend Following + Bullish News)"
                score = 90
            elif latest_rsi > 70 or sentiment_score < -0.1:
                status = "⚠️ AVOID/SELL WATCH"
                score = 20

            results.append({
                "Ticker": ticker,
                "LTP (₹)": round(latest_close, 2),
                "RSI (14)": round(latest_rsi, 2),
                "Sentiment Score": round(sentiment_score, 2),
                "Swing Status": status,
                "Action Score": score
            })

        except Exception as e:
            print(f"Skipping {ticker} due to error: {e}")
            continue

    # Format output as a highly sorted trading matrix
    final_df = pd.DataFrame(results)
    if not final_df.empty:
        final_df = final_df.sort_values(by="Action Score", ascending=False)
    return final_df

# 3. Assemble the Interactive Dashboard Interface using Gradio UI
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 📊 Pre-Market Swing Trading Dashboard (NSE India)")
    gr.Markdown("Combines **VaderSentiment Data Mining** with Technical Breakout Screeners before the 9:00 AM IST Opening Bell.") # Updated Markdown

    with gr.Row():
        scan_btn = gr.Button("🔍 Scan Watchlist Now", variant="primary")

    with gr.Row():
        output_table = gr.Dataframe(
            headers=["Ticker", "LTP (₹)", "RSI (14)", "Sentiment Score", "Swing Status", "Action Score"],
            datatype=["str", "number", "number", "number", "str", "number"],
            interactive=False
        )

    scan_btn.click(fn=scan_stocks_for_swing, outputs=output_table)

# 4. Launch the application inside Colab
# Setting share=True generates a public secure URL you can access anywhere
demo.launch(debug=True, share=True)

/tmp/ipykernel_5543/2039379348.py:104: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://65b469e0ad5598da50.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.13/dist-packages/yfinance/scrapers/history.py:204: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()
/usr/local/lib/python3.13/dist-packages/yfinance/scrapers/history.py:204: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()
/usr/local/lib/python3.13/dist-packages/yfinance/scrapers/history.py:204: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()
/usr/local/lib/python3.13/dist-packages/yfinance/scrapers/history.py:204: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()
/usr/local/lib/python3.13/dist-packages/yfinance/scrapers/history.py:204: Pandas4War